# PMSD robustness: re-run twice per dataset (expected-zero-variance check)

PMSD is fully deterministic -- expect std dev = 0.

In [ ]:
from pathlib import Path
import sys, json, time
import numpy as np
import pandas as pd
import pm4py

ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "analysis"))
sys.path.insert(0, str(ROOT / "steady_state_detection"))
sys.path.insert(0, str(ROOT / "simulation_baselines" / "PMSD-main"))

from ts_comparison import load_splits
from pmsd import run_pmsd

RESULTS = ROOT / "results"
ROBUST = ROOT / "robustness" / "pmsd"

SYNTH_DATASETS = [p.stem for p in sorted((ROOT / "data" / "synthetic").glob("*.xes")) if "recency" not in p.stem]
REAL_DATASETS = ["bpic12-a", "bpic15-1", "bpic15-2", "bpic17-o",
                 "bpic20-dom", "bpic20-int", "helpdesk", "sepsis"]

NEW_SEEDS = [43, 44]

print(f"{len(SYNTH_DATASETS)} synthetic datasets, {len(REAL_DATASETS)} real-life (ssd) datasets, seeds={NEW_SEEDS}")
print("NOTE: PMSD has zero randomness anywhere in its own code (checked: no random/seed/np.random")
print("references in simulation_baselines/PMSD-main/pmsd/*.py) -- SD-Log construction, relation")
print("discovery, and equation fitting are all deterministic; Prophet's arrival-rate fit uses its")
print("default MAP/L-BFGS optimizer (deterministic, no mcmc_samples). Both seeds below are expected")
print("to produce IDENTICAL metrics -- that IS the correct, honest result (std dev = 0), not a bug.")

In [ ]:
def run_pmsd_robustness_one(dataset: str, is_real: bool, seed: int):
    """One (dataset, seed) robustness run. Skips only when both metrics.csv and time.csv already exist."""
    sub = "ssd" if is_real else "synthetic"
    out_dir = ROBUST / sub / dataset / f"seed_{seed}"
    metrics_path = out_dir / "metrics.csv"
    time_path = out_dir / "time.csv"
    if metrics_path.exists() and time_path.exists():
        print(f"  [skip] {dataset}/seed_{seed}: metrics + time already exist")
        return

    if is_real:
        xes_path = ROOT / "data" / "real-life" / f"{dataset}.xes"
        full_log = pm4py.read_xes(str(xes_path))
        _, cc, tt, _, _, _ = load_data_for_ssd(xes_path)
        precomputed_splits = {"concurrent_cases": cc, "throughput_time": tt}
        t0 = time.perf_counter()
        res = run_pmsd(dataset, full_log, trim="ssd", is_real=True, save=False,
                       precomputed_splits=precomputed_splits)
        elapsed_s = time.perf_counter() - t0
    else:
        xes_path = ROOT / "data" / "synthetic" / f"{dataset}.xes"
        log = pm4py.read_xes(str(xes_path))
        t0 = time.perf_counter()
        res = run_pmsd(dataset, log, trim="none", is_real=False, save=False)
        elapsed_s = time.perf_counter() - t0

    m = res["metrics"]
    metrics = pd.DataFrame([
        dict(dataset=dataset, series="concurrent_cases", model="pmsd", seed=seed,
             mae=m["cc_mae"], mse=m["cc_mse"]),
        dict(dataset=dataset, series="throughput_time", model="pmsd", seed=seed,
             mae=m["tt_mae"], mse=m["tt_mse"]),
    ])
    out_dir.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(metrics_path, index=False)
    res["simulated"].to_csv(out_dir / "simulated.csv")

    time_df = pd.DataFrame([
        dict(dataset=dataset, model="pmsd", seed=seed, phase="fit", time_s=elapsed_s),
    ])
    time_df.to_csv(time_path, index=False)

    print(f"  [done] {dataset}/seed_{seed}: cc_mae={m['cc_mae']:.2f} tt_mae={m['tt_mae']:.2f}  time_s={elapsed_s:.2f}")

## Part 1: Synthetic

In [ ]:
for name in SYNTH_DATASETS:
    print(f"\n{'='*60}\n{name} (synthetic)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_pmsd_robustness_one(name, is_real=False, seed=seed)

## Part 2: SSD

In [ ]:
def load_data_for_ssd(xes_path):
    """Returns (df, cc, tt, train_df, val_df, test_df) for the ssd trim."""
    from time_series_preprocessing import Split3WayConfig, split_timeseries
    from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
    from create_prefixes_from_windows import make_three_way_split
    from ssd_trim import run_ssd_trim

    log = pm4py.read_xes(str(xes_path))

    full_cc_raw = create_concurrent_cases_timeseries(log, plot=False)
    ssd_result = run_ssd_trim(log, window_step="D")
    canonical_end = ssd_result["cutoff"] if ssd_result["cutoff"] is not None else full_cc_raw.index[-1]
    full_cc_trimmed = full_cc_raw[full_cc_raw.index <= canonical_end]
    split_cfg = Split3WayConfig(train_frac=0.70, val_frac=0.10, test_frac=0.20)
    _, _, _, train_split, val_split = split_timeseries(full_cc_trimmed, split_cfg)

    full_tt_raw = create_avg_throughtput_time_timeseries(log, plot=False)
    full_tt_trimmed = full_tt_raw[full_tt_raw.index <= canonical_end]

    def _slice(raw, trimmed):
        idx = trimmed.index
        lo = train_split.tz_convert(None) if idx.tz is None else train_split
        hi = val_split.tz_convert(None) if idx.tz is None else val_split
        return {
            "raw": raw, "trimmed": trimmed,
            "train": trimmed[idx <= lo],
            "val": trimmed[(idx > lo) & (idx <= hi)],
            "test": trimmed[idx > hi],
            "train_split": train_split, "val_split": val_split,
        }

    cc = _slice(full_cc_raw, full_cc_trimmed)
    tt = _slice(full_tt_raw, full_tt_trimmed)

    df = pm4py.convert_to_dataframe(log)
    df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], utc=True)
    df = df.dropna(subset=["case:concept:name"])
    _cols = {"case:concept:name": "caseid", "concept:name": "task",
             "lifecycle:transition": "event_type", "time:timestamp": "end_timestamp"}
    _cols["org:resource" if "org:resource" in df.columns else "org:group"] = "user"
    df = df.rename(columns=_cols)
    df["task"] = df["task"].fillna("unk")
    df["user"] = df["user"].fillna("unk")

    train_, val_, test_ = make_three_way_split(
        df, case_col="caseid", time_col="end_timestamp",
        train_split=cc["train_split"], val_split=cc["val_split"], full_traces=True,
    )
    return df, cc, tt, train_, val_, test_

In [ ]:
for name in REAL_DATASETS:
    print(f"\n{'='*60}\n{name} (ssd)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_pmsd_robustness_one(name, is_real=True, seed=seed)